# 06 · Диагностика входных кропов и инференса
Тест Авито используется только для размеров и проверки файлов. Не создаём метки и не показываем тестовые кропы. Effective height — высота всего кропа после resize, а не измеренная высота букв.

In [ ]:
from pathlib import Path


def find_project_root(start=None):
    from pathlib import Path
    import os

    current = Path(start or Path.cwd()).resolve()
    explicit = os.environ.get("AVITO_PROJECT_ROOT")
    candidates = (
        [Path(explicit).expanduser().resolve()] if explicit else [current, *current.parents]
    )
    for candidate in candidates:
        marker = (candidate / "avito_project.json").is_file()
        layout = (candidate / "notebooks" / "_shared.ipynb").is_file() and (
            candidate / "requirements.txt"
        ).is_file()
        if marker or layout:
            return candidate
    raise RuntimeError(
        f"Не найден корень Avito_test. Текущая папка: {current}. "
        "Перенеси notebooks/ целиком, requirements.txt и avito_project.json в папку проекта. "
        "На VM выполни в отдельной ячейке: %cd /home/slomauh/ml/Avito_test . "
        "Для другого расположения задай переменную окружения AVITO_PROJECT_ROOT. "
        + (f"Сейчас AVITO_PROJECT_ROOT={explicit!r}; проверь этот путь." if explicit else "")
    )


ROOT = find_project_root()
get_ipython().run_line_magic("run", str(ROOT / "notebooks" / "_shared.ipynb"))
print("Проект:", ROOT)

get_ipython().run_line_magic("run", str(ROOT / "notebooks/_diagnostics.ipynb"))


In [ ]:
MANIFEST = ROOT / "data/processed/hiertext_full/manifest.csv"
CHECKPOINT = ROOT / "runs/mobilenet_v1/best.pt"
TEST_DIR = ROOT / "data/test"
SAMPLE_SUBMISSION = ROOT / "data/sample_submission.csv"
EXPECTED_COUNT = 20000
OUTPUT_DIR = ROOT / "runs/diagnostics_v1"  # новое имя для повторного запуска
HEIGHT, WIDTH = 48, 192
MAX_EXTERNAL = 20000


In [ ]:
manifest = load_manifest(MANIFEST)
ids, paths = test_files(TEST_DIR, SAMPLE_SUBMISSION, EXPECTED_COUNT)
model, state = load_checkpoint(CHECKPOINT, torch.device("cpu"))
torch.set_num_threads(4)
if state["data_identity"]["manifest_sha256"] != sha256_file(MANIFEST):
    raise ValueError("Manifest отличается от обучающего")
if (state["config"]["height"], state["config"]["width"]) != (HEIGHT, WIDTH):
    raise ValueError("Разрешение отличается от checkpoint")
if OUTPUT_DIR.exists():
    raise FileExistsError("Выбери новый OUTPUT_DIR")
OUTPUT_DIR.mkdir(parents=True)
external = manifest[manifest.split.isin(["train", "dev"])].sort_values("crop_id")
external = external.sample(min(MAX_EXTERNAL, len(external)), random_state=42)
test_frame = pd.DataFrame({"crop_id": ids, "path": [str(p) for p in paths]})
external.to_csv(OUTPUT_DIR / "external_selection.csv", index=False)
test_frame.to_csv(OUTPUT_DIR / "test_mapping.csv", index=False)
tables = []
for name, frame in [("hiertext", external), ("avito", test_frame)]:
    stats = image_stats(frame, HEIGHT, WIDTH)
    stats["dataset"] = name
    tables.append(stats)
stats = pd.concat(tables, ignore_index=True)
stats.to_csv(OUTPUT_DIR / "geometry.csv", index=False)
summary = stats.groupby("dataset")[
    ["width", "height", "aspect", "effective_height", "occupied_fraction"]
].describe(percentiles=[0.05, 0.5, 0.95])
summary.to_csv(OUTPUT_DIR / "summary.csv")
display(summary)
display(
    stats.assign(height_below_12=stats.effective_height < 12)
    .groupby("dataset")
    .height_below_12.mean()
)
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name, g in stats.groupby("dataset"):
    axs[0].hist(
        np.log2(g.aspect), bins=np.linspace(-6, 10, 65), density=True, alpha=0.4, label=name
    )
    axs[1].hist(g.effective_height, bins=np.arange(0.5, 49.5), density=True, alpha=0.4, label=name)
axs[0].set_xlabel("log2(width/height)")
axs[1].set_xlabel("Высота кропа после resize")
for ax in axs:
    ax.legend()
plt.show()


In [ ]:
frame = select_dev(manifest, 32)
cfg = state["config"].copy()
cfg["batch_size"] = 16
pred = predict_loader(model, make_loader(frame, cfg), torch.device("cpu"), rotated=True)
ds = CropDataset(frame, HEIGHT, WIDTH)
for i in range(len(ds)):
    x, y, sid = ds[i]
    expected = open_rgb(ROOT / frame.iloc[i // 2].path)
    if i % 2:
        expected = expected.transpose(Image.Transpose.ROTATE_180)
    assert y == i % 2 and sid.endswith("__r" + str(180 * (i % 2)))
    assert torch.equal(x, letterbox(expected, HEIGHT, WIDTH))
report = {}
for tta in [False, True]:
    p = predict_images(
        model,
        [ROOT / p for p in frame.path],
        cfg,
        {"temperature": 1.0, "tta": tta},
        torch.device("cpu"),
        16,
    )
    reference = probabilities(pred, 1.0, tta)[::2]
    error = float(np.max(np.abs(p - reference)))
    if error > 1e-5:
        raise AssertionError(f"Пути инференса различаются: {error}")
    pair = probabilities(pred, 1.0, tta).reshape(-1, 2)
    report[str(tta)] = {
        "path_max_abs_error": error,
        "rotation_complement_mean_error": float(np.abs(pair.sum(1) - 1).mean()),
        **metrics(pred.y, probabilities(pred, 1.0, tta)),
    }
pixel_errors = []
for path in frame.path:
    im = open_rgb(ROOT / path)
    pixel_errors.append(
        float(
            (
                letterbox(im.transpose(Image.Transpose.ROTATE_180), HEIGHT, WIDTH)
                - torch.flip(letterbox(im, HEIGHT, WIDTH), (-2, -1))
            )
            .abs()
            .max()
        )
    )
report["letterbox_rotation_max_difference"] = max(pixel_errors)
report["note"] = (
    "Нечётные поля сдвигаются на 1 px при повороте. Поэтому TTA после letterbox не гарантирует точное p(x)+p(rot(x))=1 для исходных файлов. Предобработку старой модели не меняем."
)
report["identity"] = {
    "checkpoint_sha256": sha256_file(CHECKPOINT),
    "manifest_sha256": sha256_file(MANIFEST),
    "sample_sha256": sha256_file(SAMPLE_SUBMISSION),
    "environment": environment(),
}
save_json(OUTPUT_DIR / "checks.json", report)
print(json.dumps(report, ensure_ascii=False, indent=2))
fig, axs = plt.subplots(4, 3, figsize=(12, 8))
for row, (_, r) in zip(axs, frame.head(4).iterrows()):
    im = open_rgb(ROOT / r.path)
    x = letterbox(im, HEIGHT, WIDTH).permute(1, 2, 0).numpy()
    x = np.clip(x * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
    for ax, img, title in zip(
        row,
        [im, x, im.transpose(Image.Transpose.ROTATE_180)],
        ["Внешний upright: y=0", "Вход модели", "Поворот: y=1"],
    ):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()


Если проверки упали — не запускай обучение, сначала исправь причину. Низкое качество на 32 кропах не является общей оценкой. Пришли checks.json и summary.csv; различия размеров сами по себе не доказывают причину падения качества.